In [ ]:
%sql
-- Explicit operator parameters; this notebook has no scheduled activation.
-- stage -> inspect/hash/approve -> activate. Each is a separate authorized job run.
SELECT assert_true(:phase IN ('stage','activate'),'phase must be stage or activate'),
 assert_true(:release_prefix RLIKE '^(openalex[.]funders[.]nm1013_|openalex_dev[.]rohan_lab[.]ff1013_)[a-z0-9_]+_$', 'release prefix invalid'),
 assert_true(:catalogue IN ('openalex.funders.funders') OR :catalogue RLIKE '^openalex_dev[.]rohan_lab[.]ff1013_', 'catalogue fence'),
 assert_true(:lookup = 'openalex.funders.funder_name_lookup' OR :lookup RLIKE '^openalex_dev[.]rohan_lab[.]ff1013_', 'lookup fence');
SELECT assert_true((:registry_prefix = 'openalex.funders.' AND :catalogue = 'openalex.funders.funders' AND :lookup='openalex.funders.funder_name_lookup' AND :release_prefix LIKE 'openalex.funders.nm1013_%') OR (:registry_prefix RLIKE '^openalex_dev[.]rohan_lab[.]ff1013_[a-z0-9_]+_$' AND :catalogue RLIKE '^openalex_dev[.]rohan_lab[.]ff1013_' AND :lookup RLIKE '^openalex_dev[.]rohan_lab[.]ff1013_' AND :release_prefix RLIKE '^openalex_dev[.]rohan_lab[.]ff1013_'),'all bindings must share the production or dev fence');


In [ ]:
%sql
BEGIN
IF :phase = 'stage' THEN
-- CREATE (no REPLACE) makes this release immutable; use a fresh release id to restage.
CREATE TABLE IDENTIFIER(:release_prefix || 'catalogue_before') AS SELECT * FROM IDENTIFIER(:catalogue);
CREATE TABLE IDENTIFIER(:release_prefix || 'registry_before') AS SELECT * FROM IDENTIFIER(:registry_prefix || 'funder');
CREATE TABLE IDENTIFIER(:release_prefix || 'identifiers_before') AS SELECT * FROM IDENTIFIER(:registry_prefix || 'funder_id');
END IF;
END;


In [ ]:
%sql
BEGIN
IF :phase = 'stage' THEN
-- name_aliases: exact UTF-8 keys from name_aliases.csv; no field or DOI edits.
-- VALUES is the SQL-cell mirror of the reviewed CSV (checked by validation).
CREATE TABLE IDENTIFIER(:release_prefix || 'aliases') AS
SELECT decode(unhex(name_hex),'UTF-8') name_key, funder_id, rule
FROM VALUES
('6e6174696f6e616c2068656172742c206c756e672c20616e6420626c6f6f6420696e7374697475746520286e686c626929',4320337338L,'reviewed'),
('6d696e6973747279206f6620656475636174696f6e206f66207468652070656f706c652661706f733b732072657075626c6963206f66206368696e61',4320321106L,'reviewed'),
('6e6174696f6e616c206e61747572616c20736369656e636520666f756e646174696f6e206f66206368696e6120286e73666329',4320321001L,'reviewed'),
('6d696e6973747279206f6620736369656e636520616e6420746563686e6f6c6f6779206f66207468652070656f706c652661706f733b732072657075626c6963206f66206368696e61',4320321540L,'reviewed'),
('666f75727468206672616d65776f726b2070726f6772616d6d65',4320333069L,'reviewed'),
('686873207c206e6174696f6e616c20696e7374697475746573206f66206865616c7468',4320332161L,'reviewed'),
('6d657874207c206a6170616e20736f636965747920666f72207468652070726f6d6f74696f6e206f6620736369656e6365',4320334764L,'reviewed'),
('6d696e6973747279206f6620656475636174696f6e2c2063756c747572652c2073706f7274732c20736369656e63652c20616e6420746563686e6f6c6f6779',4320320912L,'reviewed'),
('7a68656a69616e672070726f76696e6365206e61747572616c20736369656e636520666f756e646174696f6e',4320338464L,'reviewed'),
('7369787468206672616d65776f726b2070726f6772616d6d65206f6620746865206575726f7065616e20636f6d6d697373696f6e',4320333067L,'reviewed'),
('686873207c206e6968207c206e6174696f6e616c20696e73746974757465206f6620616c6c6572677920616e6420696e66656374696f7573206469736561736573',4320337355L,'reviewed'),
('752e732e206465706172746d656e74206f66206865616c746820262068756d616e207365727669636573207c206e6174696f6e616c20696e7374697475746573206f66206865616c7468',4320332161L,'reviewed'),
('726f636865',4320307779L,'reviewed'),
('686873207c206e6968207c206e6174696f6e616c20696e73746974757465206f662067656e6572616c206d65646963616c20736369656e636573',4320337354L,'reviewed'),
('6b657920746563686e6f6c6f677920726573656172636820616e6420646576656c6f706d656e742070726f6772616d206f66207368616e646f6e672070726f76696e6365',4320333596L,'reviewed'),
('7369636875616e2070726f76696e6369616c20736369656e636520616e6420746563686e6f6c6f677920737570706f72742070726f6772616d',4320333335L,'reviewed'),
('6b6f726561206d696e6973747279206f6620736369656e636520616e6420696374',4320328359L,'reviewed'),
('6e61747572616c20736369656e636520666f756e646174696f6e206f66207469616e6a696e206d756e69636970616c697479',4320323993L,'reviewed'),
('6775616e67646f6e6720736369656e636520616e6420746563686e6f6c6f6779206465706172746d656e74',4320324202L,'reviewed'),
('616c6672656420776567656e657220696e737469747574652c2068656c6d686f6c747a2063656e74726520666f7220706f6c617220616e64206d6172696e652072657365617263682c206272656d6572686176656e',4320321790L,'reviewed'),
('6832303230206d6172696520736b6c6f646f77736b612d637572696520616374696f6e73',4320338337L,'reviewed'),
('636f6e73656c686f206e6163696f6e616c20646520646573656e766f6c76696d656e746f206369656e743f6669636f2065207465636e6f6c3f6769636f',4320322025L,'reviewed'),
('752e732e206465706172746d656e74206f66206865616c746820262068756d616e207365727669636573207c206e6968207c206e6174696f6e616c20696e73746974757465206f662067656e6572616c206d65646963616c20736369656e636573',4320337354L,'reviewed'),
('6465706172746d656e74206f662062696f746563686e6f6c6f6779202c206d696e6973747279206f6620736369656e636520616e6420746563686e6f6c6f6779',4320320717L,'reviewed'),
('7374617465206f662073616f207061756c6f20726573656172636820666f756e646174696f6e',4320320997L,'reviewed'),
('752e732e206465706172746d656e74206f66206865616c746820262068756d616e207365727669636573207c206e6968207c206e6174696f6e616c2063616e63657220696e73746974757465',4320337351L,'reviewed'),
('636f6f7264656e613f3f6f20646520617065726665693f6f616d656e746f20646520706573736f616c206465206e3f76656c207375706572696f72',4320321091L,'reviewed'),
('6265696a696e67206d756e69636970616c20636f6d6d697373696f6e206f6620656475636174696f6e',4320321572L,'reviewed'),
('68656c6d686f6c747a2d67656d65696e736368616674',4320320876L,'reviewed'),
('736576656e7468206672616d65776f726b2070726f6772616d6d65206f6620746865206575726f7065616e20636f6d6d697373696f6e',4320333065L,'reviewed'),
('636f6f7264696e6174696f6e206f662068696768657220656475636174696f6e20706572736f6e6e656c20696d70726f76656d656e74',4320321091L,'reviewed'),
('68756265692070726f76696e6365206e61747572616c20736369656e636520666f756e646174696f6e',4320322186L,'reviewed'),
('686873207c206e6968207c206e6174696f6e616c2068656172742c206c756e672c20616e6420626c6f6f6420696e73746974757465',4320337338L,'reviewed'),
('6e6174696f6e616c206e61747572616c20736369656e636520666f756e646174696f6e73206f66206368696e61',4320321001L,'reviewed'),
('6d6f7374207c206e6174696f6e616c206e61747572616c20736369656e636520666f756e646174696f6e206f66206368696e61',4320321001L,'reviewed'),
('7363687765697a6572697363686572206e6174696f6e616c666f6e6473207a757220663f72646572756e67206465722077697373656e7363686166746c696368656e20666f72736368756e67',4320320924L,'reviewed'),
('646570617274616d656e746f2061646d696e69737472617469766f206465206369656e6369612c207465636e6f6c6f67c3ad61206520696e6e6f76616369c3b36e',4320309955L,'reviewed'),
('6563207c20686f72697a6f6e2032303230206672616d65776f726b2070726f6772616d6d65',4320332999L,'reviewed'),
('6775616e677a686f75206d756e69636970616c20736369656e636520616e6420746563686e6f6c6f67792070726f6772616d206b65792070726f6a65637473',4320334898L,'reviewed'),
('6c69616f6e696e672070726f76696e6369616c206e61747572616c20736369656e636520666f756e646174696f6e',4320323086L,'reviewed'),
('6865696c6f6e676a69616e672070726f76696e6365206e61747572616c20736369656e636520666f756e646174696f6e',4320323085L,'reviewed'),
('686873207c206e6968207c206e6174696f6e616c2063616e63657220696e73746974757465',4320337351L,'reviewed'),
('736861616e78692070726f76696e6365206b657920726573656172636820616e6420646576656c6f706d656e742070726f6a65637473',4320336350L,'reviewed'),
('752e732e206465706172746d656e74206f66206865616c746820262068756d616e207365727669636573207c206e6968207c206e6174696f6e616c20696e73746974757465206f66206d656e74616c206865616c7468',4320337346L,'reviewed'),
('676f757665726e656d656e742064752063616e616461207c206e61747572616c20736369656e63657320616e6420656e67696e656572696e6720726573656172636820636f756e63696c206f662063616e616461',4320334593L,'reviewed'),
('6b6f726561206d696e6973747279206f6620747261646520696e64757374727920616e6420656e65726779',4320321681L,'reviewed'),
('686873207c206e6968207c206e6174696f6e616c20696e73746974757465206f6620646961626574657320616e642064696765737469766520616e64206b69646e6579206469736561736573',4320337357L,'reviewed'),
('676f757665726e656d656e742064752063616e616461207c2063616e616469616e20696e7374697475746573206f66206865616c7468207265736561726368',4320334506L,'reviewed'),
('6e6174696f6e616c206e61747572616c20736369656e636520666f756e646174696f6e206f66206368696e61206e6174696f6e616c206f75747374616e64696e6720796f75746820736369656e63652066756e642070726f6a656374',4320333688L,'reviewed'),
('636f756e63696cc2a06f66c2a0736369656e746966696320616e6420696e647573747269616c2072657365617263682c20696e646961',4320320721L,'reviewed'),
('686873207c206e6968207c206e6174696f6e616c20696e73746974757465206f66206e6575726f6c6f676963616c206469736f726465727320616e64207374726f6b65',4320337359L,'reviewed'),
('62756e6465736d696e697374657269756d20663f722062696c64756e6720756e6420666f72736368756e67',4320321114L,'reviewed'),
('6a696c696e2070726f76696e6369616c20736369656e746966696320616e6420746563686e6f6c6f676963616c20646576656c6f706d656e742070726f6772616d',4320335953L,'reviewed'),
('6e6174696f6e616c20696e7374697475746573206f66206865616c7468206e6174696f6e616c2063616e63657220696e73746974757465',4320337351L,'reviewed'),
('746865206e61747572616c20736369656e636520666f756e646174696f6e206f66206368696e61',4320321001L,'reviewed'),
('66756e64613f3f6f20646520616d7061726f203f20706573717569736120646f2065737461646f20646520733f6f207061756c6f',4320320997L,'reviewed'),
('6e6174696f6e616c206165726f737061636520736369656e636520666f756e646174696f6e206f66206368696e61',4320326181L,'reviewed'),
('74616b65646120706861726d61636575746963616c20636f6d70616e79',4320310465L,'reviewed'),
('6e6174696f6e616c20736369656e636520636f756e63696c2074616977616e',4320321040L,'reviewed'),
('766574656e736b617073723f646574',4320322581L,'reviewed'),
('6d696e6973746572696f206465206369656e636961206520696e6e6f766163693f6e',4320322930L,'reviewed'),
('7263756b207c206d65646963616c20726573656172636820636f756e63696c',4320334626L,'reviewed'),
('756e6976657273697479206772616e747320636f6d6d6974746565207265736561726368206772616e747320636f756e63696c',4320321592L,'reviewed'),
('6465706172746d656e74206f66206865616c7468207c206e6174696f6e616c206865616c746820616e64206d65646963616c20726573656172636820636f756e63696c',4320334705L,'reviewed'),
('686f72697a6f6e2032303230206f6620746865206575726f7065616e20636f6d6d697373696f6e',4320335254L,'reviewed'),
('6a696c696e2070726f76696e6369616c20736369656e636520616e6420746563686e6f6c6f6779206465706172746d656e74',4320327282L,'reviewed'),
('6368696e6573652061636164656d79206f6620736369656e63657320796f75746820696e6e6f766174696f6e2070726f6d6f74696f6e206173736f63696174696f6e',4320322847L,'reviewed'),
('6d616c6179736961206d696e6973747279206f662068696768657220656475636174696f6e',4320321709L,'reviewed'),
('66756e64613f3f6f207061726120612063693f6e63696120652061207465636e6f6c6f676961',4320334779L,'reviewed'),
('7263756b207c20656e67696e656572696e6720616e6420706879736963616c20736369656e63657320726573656172636820636f756e63696c',4320334627L,'reviewed'),
('6d696e6973747279206f6620656475636174696f6e20796f75746820616e642073706f727473206f662074686520637a6563682072657075626c6963',4320321005L,'reviewed'),
('6a6f686e736f6e20616e64206a6f686e736f6e',4320332194L,'reviewed'),
('752e732e206465706172746d656e74206f66206865616c746820262068756d616e207365727669636573207c206e6968207c206e6174696f6e616c20696e73746974757465206f66206e6575726f6c6f676963616c206469736f726465727320616e64207374726f6b65',4320337359L,'reviewed'),
('6775697a686f752070726f76696e6369616c20736369656e636520616e6420746563686e6f6c6f6779206465706172746d656e74',4320322272L,'reviewed'),
('6167c3a86e636961206465206765737469c3b32064e28099616a75747320756e697665727369746172697320692064652072656365726361',4320334830L,'reviewed'),
('752e732e206465706172746d656e74206f66206865616c746820262068756d616e207365727669636573207c206e6968207c206e6174696f6e616c20696e73746974757465206f6620616c6c6572677920616e6420696e66656374696f7573206469736561736573',4320337355L,'reviewed'),
('6d696e6973746572696f206465206369656e6369612c20696e6e6f766163693f6e207920756e6976657273696461646573',4320315062L,'reviewed'),
('6b6f726561206d696e6973747279206f6620656475636174696f6e',4320321408L,'reviewed'),
('636861726974c3a9202d20756e69766572736974c3a474736d6564697a696e206265726c696e',4320321526L,'reviewed'),
('752e732e206465706172746d656e74206f66206865616c746820262068756d616e207365727669636573207c206e6968207c206e6174696f6e616c2068656172742c206c756e672c20616e6420626c6f6f6420696e73746974757465',4320337338L,'reviewed'),
('636f6e73656c686f206e6163696f6e616c20646520646573656e766f6c76696d656e746f206369656e74c3a3c2ad6669636f2065207465636e6f6cc3a3c2b36769636f',4320322025L,'reviewed'),
('6465706172746d656e74206f6620736369656e636520616e6420746563686e6f6c6f6779206f662068656e616e2070726f76696e6365',4320326718L,'reviewed'),
('6e6174696f6e616c206e61747572616c20736369656e636520666f756e646174696f6e206f66206368696e612028636e29',4320321001L,'reviewed'),
('6665646572616c206d696e6973747279206f6620656475636174696f6e20616e6420726573656172636820626f6e6e206f6666696365',4320321114L,'reviewed'),
('746865206368696e6120706f7374646f63746f72616c20736369656e636520666f756e646174696f6e',4320321543L,'reviewed'),
('7469616e6a696e206d756e69636970616c20736369656e636520616e6420746563686e6f6c6f67792070726f6772616d',4320336756L,'reviewed'),
('6563207c206575726f7065616e20726573656172636820636f756e63696c',4320334678L,'reviewed'),
('7263756b207c2062696f746563686e6f6c6f677920616e642062696f6c6f676963616c20736369656e63657320726573656172636820636f756e63696c',4320334629L,'reviewed'),
('636f6c756d62696120756e6976657273697479',4320309111L,'reviewed'),
('676f7665726e6d656e74206f66206a6170616e206d696e6973747279206f6620656475636174696f6e2063756c747572652073706f72747320736369656e636520616e6420746563686e6f6c6f6779',4320320912L,'reviewed'),
('68756d616e697469657320616e6420736f6369616c20736369656e63652066756e64206f66206d696e6973747279206f6620656475636174696f6e206f66207468652070656f706c6527732072657075626c6963206f66206368696e61',4320335982L,'reviewed'),
('666f6e647320646520726563686572636865206475207175c3a9626563202d206e617475726520657420746563686e6f6c6f67696573',4320334841L,'reviewed'),
('65737061c3b161206d696e6973746572696f206465206369656e63696120696e6e6f76616369c3b36e207920756e6976657273696461646573',4320315062L,'reviewed'),
('746865206e6174696f6e616c206e617475726520736369656e636520666f756e646174696f6e206f66206368696e61',4320321001L,'reviewed'),
('756e6976657273697479206f662063616c69666f726e6961206265726b656c6579',4320309411L,'reviewed'),
('6d696e69737465726f2064656c6c2661706f733b756e69766572736974c3a020652064656c6c612072696365726361',4320331528L,'reviewed'),
('64757463682063616e63657220736f6369657479',4320322777L,'reviewed'),
('6d696e69737465726f2064656c6c3f69737472757a696f6e652c2064656c6c3f756e697665727369743f20652064656c6c612072696365726361',4320321873L,'reviewed'),
('636f6e73656c686f206e6163696f6e616c20646520646573656e766f6c76696d656e746f206369656e74696669636f2065207465636e6f6c6f6769636f',4320322025L,'reviewed'),
('6e61747572616c20736369656e636520666f756e646174696f6e206f66207869616d656e206d756e69636970616c697479',4320316884L,'reviewed'),
('6772616e746f763f206167656e74757261206365736b3f2072657075626c696b79',4320321006L,'reviewed'),
('6b696e67206b68616c696420756e6976657273697479206465616e73686970206f6620736369656e7469666963207265736561726368',33954902L,'reviewed'),
('737061696e206d696e6973747279206f6620736369656e636520696e6e6f766174696f6e20616e6420756e69766572736974696573',4320315062L,'reviewed'),
('686f72697a6f6e2032303230206d6172696520736bc5826f646f77736b612d637572696520616374696f6e73',4320338337L,'reviewed'),
('6e6968206e6174696f6e616c20696e73746974757465206f662067656e6572616c206d65646963616c20736369656e636573',4320337354L,'reviewed'),
('6e61747572616c20736369656e636520666f756e646174696f6e206f662071696e6764616f206d756e69636970616c697479',4320328636L,'reviewed'),
('6e61747572616c20736369656e636520666f756e646174696f6e206f66207368656e7a68656e206d756e69636970616c697479',4320316880L,'reviewed'),
('752e732e206465706172746d656e74206f66206865616c746820262068756d616e207365727669636573207c206e6968207c206e6174696f6e616c20696e73746974757465206f6e206167696e67',4320337337L,'reviewed'),
('756b20726573656172636820616e6420696e6e6f766174696f6e206d65646963616c20726573656172636820636f756e63696c',4320334626L,'reviewed'),
('636f6f7264656e61c3a3c2a7c3a3c2a36f2064652061706572666569c3a3c2a76f616d656e746f20646520706573736f616c206465206ec3a3c2ad76656c207375706572696f72',4320321091L,'reviewed'),
('6d696e6973747279206f6620656475636174696f6e206f66207468652070656f706c65e28099732072657075626c6963206f66206368696e61',4320321106L,'reviewed'),
('6d6f7374207c206e6174696f6e616c206b657920726573656172636820616e6420646576656c6f706d656e742070726f6772616d206f66206368696e61',4320335777L,'reviewed'),
('6e61747572616c20736369656e636520666f756e646174696f6e206f66206865696c6f6e676a69616e672070726f76696e6365206f66206368696e61',4320323085L,'reviewed'),
('6d696e6973747279206f66206167726963756c7475726520616e6420727572616c2061666661697273206f66207468652070656f706c652661706f733b732072657075626c6963206f66206368696e61',4320327287L,'reviewed'),
('7265736561726368206772616e747320636f756e63696c2c20756e6976657273697479206772616e747320636f6d6d69747465652c20686f6e67206b6f6e67',4320321592L,'reviewed'),
('752e732e206465706172746d656e74206f66206865616c746820262068756d616e207365727669636573207c206e6968207c206e6174696f6e616c20696e73746974757465206f6620646961626574657320616e642064696765737469766520616e64206b69646e6579206469736561736573',4320337357L,'reviewed'),
('752e732e206465706172746d656e74206f6620656e657267792028646f6529',4320306084L,'reviewed'),
('67616e73752070726f76696e6369616c206e61747572616c20736369656e636520666f756e646174696f6e',4320322880L,'reviewed'),
('686873207c206e6968207c206e6174696f6e616c20696e73746974757465206f6e206167696e67',4320337337L,'reviewed'),
('77696c6c69616d20742e206772616e7420666f756e646174696f6e',4320306360L,'reviewed'),
('6d696e69737465726f2064656c6c27756e697665727369743f20652064656c6c612072696365726361',4320331528L,'reviewed'),
('676f7665726e6d656e74206f662073696e6761706f7265206d696e6973747279206f6620656475636174696f6e',4320320751L,'reviewed'),
('6d657874207c206a6170616e20736369656e636520616e6420746563686e6f6c6f6779206167656e6379',4320334789L,'reviewed'),
('6d696e69737465727374766f203f6b6f6c7374763f2c206d6c3f64653f6520612074656c6f763f63686f7679',4320321005L,'reviewed'),
('7368616e67686169206d756e69636970616c69747920736369656e636520616e6420746563686e6f6c6f677920636f6d6d697373696f6e',4320321885L,'reviewed'),
('6167656e636961206573746174616c20646520696e7665737469676163693f6e',4320335598L,'reviewed'),
('696e646961206d696e6973747279206f6620736369656e6365202620746563686e6f6c6f6779206465706172746d656e74206f6620736369656e636520616e6420746563686e6f6c6f6779',4320320719L,'reviewed'),
('686873207c206e6968207c206e6174696f6e616c20696e73746974757465206f66206d656e74616c206865616c7468',4320337346L,'reviewed'),
('68756d616e697469657320616e6420736f6369616c20736369656e63652066756e64206f66206d696e6973747279206f6620656475636174696f6e206f66207468652070656f706c652661706f733b732072657075626c6963206f66206368696e61',4320335982L,'reviewed'),
('6e6174696f6e616c206e61747572616c20736369656e746966696320666f756e646174696f6e206f66206368696e61',4320321001L,'reviewed'),
('756b20656e67696e656572696e6720616e6420706879736963616c20736369656e63657320726573656172636820636f756e63696c',4320334627L,'reviewed'),
('6861696e616e2070726f76696e6365206e61747572616c20736369656e636520666f756e646174696f6e',4320322866L,'reviewed'),
('73616e6d696e672070726f6a656374206f66206d65646963696e6520696e207368656e7a656e206d756e69636970616c697479',4320335767L,'reviewed'),
('736861616e78692070726f76696e6365206465706172746d656e74206f6620736369656e636520616e6420746563686e6f6c6f6779',4320327224L,'reviewed'),
('752e732e206465706172746d656e74206f66206865616c746820262068756d616e207365727669636573207c206e6968207c206e6174696f6e616c20696e73746974757465206f6e2064727567206162757365',4320337347L,'reviewed'),
('686873207c206e6174696f6e616c20696e7374697475746573206f66206865616c746820286e696829',4320332161L,'reviewed'),
('743f726b6979652062696c696d73656c2076652074656b6e6f6c6f6a696b20617261737469726d61206b7572756d75',4320322626L,'reviewed'),
('736d69746820616e64206e6570686577',4320310956L,'reviewed'),
('6e6174696f6e616c206e61747572616c20736369656e636520666f756e646174696f6e206f660a0909090909096368696e61',4320321001L,'reviewed'),
('6e6f7267657320666f72736b6e696e6773723f64',4320323299L,'reviewed'),
('6a706220666f756e646174696f6e',4320309814L,'reviewed'),
('6e6174696f6e616c20696e73746974757465206f6620616c6c6572677920616e6420696e66656374696f7573206469736561736573206469766973696f6e206f6620696e7472616d7572616c207265736561726368',4320309123L,'reviewed'),
('7263756b207c206e61747572616c20656e7669726f6e6d656e7420726573656172636820636f756e63696c',4320334631L,'reviewed'),
('636f6f7264656e61c3a7c3a36f2064652061706572666569c3a76f616d656e746f20646520706573736f616c206465206ec3ad76656c207375706572696f72202d2062726173696c',4320321091L,'reviewed'),
('746865206e6174696f6e616c20726573656172636820666f756e646174696f6e206f66206b6f726561',4320322120L,'reviewed'),
('6b656d656e74657269616e2072697365742c2074656b6e6f6c6f67692064616e2070656e646964696b616e2074696e6767692072657075626c696b20696e646f6e65736961',4320325829L,'reviewed'),
('636f6e73656a6f206e6163696f6e616c20646520696e7665737469676163696f6e6573206369656e743f6669636173207920743f636e69636173',4320321594L,'reviewed'),
('6772616e746f76c3a1206167656e74757261206365736bc3a92072657075626c696b79',4320321006L,'reviewed'),
('7369636875616e2070726f76696e6365206465706172746d656e74206f6620736369656e636520616e6420746563686e6f6c6f6779',4320322922L,'reviewed'),
('6361726c6f73206368616761732066696c686f20666f756e646174696f6e20666f7220726573656172636820737570706f7274206f662072696f206465206a616e6569726f207374617465',4320322749L,'reviewed'),
('6d657874207c206a7374207c20636f726520726573656172636820666f722065766f6c7574696f6e616c20736369656e636520616e6420746563686e6f6c6f6779',4320338075L,'reviewed'),
('7368656e7a68656e206d756e69636970616c2066756e64616d656e74616c2072657365617263682070726f6772616d',4320329791L,'reviewed'),
('616c7a6865696d6572e2809973206173736f63696174696f6e',4320306219L,'reviewed'),
('6d696e61732067657261697320737461746520666f756e646174696f6e206f6620737570706f727420746f20746865207265736561726368',4320322980L,'reviewed'),
('686873207c206e6968207c206e6174696f6e616c20696e73746974757465206f6620646961626574657320616e642064696765737469766520616e64206b69646e657920646973656173657320286e6964646b29',4320337357L,'reviewed'),
('66756e646163616f20706172612061206369656e63696120652061207465636e6f6c6f676961',4320334779L,'reviewed'),
('6e6174696f6e616c206865617274206c756e6720616e6420626c6f6f6420696e73746974757465206469766973696f6e206f6620696e7472616d7572616c207265736561726368',4320338421L,'reviewed'),
('276c612063616978612720666f756e646174696f6e',4320312058L,'reviewed'),
('686873207c206e6968207c206e6174696f6e616c2068656172742c206c756e672c20616e6420626c6f6f6420696e7374697475746520286e68626c6929',4320337338L,'reviewed'),
('7363687765697a65726973636865206e6174696f6e616c666f6e6473207a75722066c3b672646572756e67206465722077697373656e7363686166746c696368656e20666f72736368756e67',4320320924L,'reviewed'),
('616e6875692070726f76696e6365206465706172746d656e74206f6620656475636174696f6e',4320326664L,'reviewed'),
('6a69616e6773752070726f76696e6365206465706172746d656e74206f6620736369656e636520616e6420746563686e6f6c6f6779',4320325437L,'reviewed'),
('6368696e657365206165726f6e6175746963616c2065737461626c6973686d656e74206165726f6e6175746963616c20736369656e636520666f756e646174696f6e',4320322857L,'reviewed'),
('696d616d206d6f68616d6d65642069626e20736175642069736c616d696320756e6976657273697479206465616e73686970206f6620736369656e7469666963207265736561726368',2666706692L,'reviewed'),
('636f6e73656a6f207375706572696f7220646520696e7665737469676163696f6e6573206369656e74696669636173',4320334857L,'reviewed'),
('752e732e206465706172746d656e74206f66206865616c746820262068756d616e207365727669636573207c206e6968207c2065756e696365206b656e6e6564792073687269766572206e6174696f6e616c20696e73746974757465206f66206368696c64206865616c746820616e642068756d616e20646576656c6f706d656e74',4320337611L,'reviewed'),
('72657075626c6963206f6620736572626961206d696e6973747279206f6620656475636174696f6e20736369656e636520616e6420746563686e6f6c6f676963616c20646576656c6f706d656e74',4320322729L,'reviewed'),
('646f65207c207363207c20626173696320656e6572677920736369656e636573',4320337480L,'reviewed'),
('736369656e636520616e6420746563686e6f6c6f6779206d616a6f722070726f6a656374206f6620696e6e6572206d6f6e676f6c6961206175746f6e6f6d6f757320726567696f6e206f66206368696e61',4320331106L,'reviewed'),
('6e61747572616c20736369656e636520666f756e646174696f6e206f663f68756e616e2070726f76696e6365',4320322843L,'reviewed'),
('6e6174696f6e616c20736369656e636520666f756e646174696f6e206469726563746f7261746520666f7220656e67696e656572696e67',4320332170L,'reviewed'),
('676f7665726e6d656e74206f66206a6170616e206d696e6973747279206f66206865616c7468206c61626f757220616e642077656c66617265',4320321945L,'reviewed'),
('756e6974656420737461746573206465706172746d656e74206f6620646566656e7365207c20756e69746564207374617465732061697220666f726365207c2061666d63207c2061697220666f726365206f6666696365206f6620736369656e7469666963207265736561726368',4320338279L,'reviewed'),
('756b20726573656172636820616e6420696e6e6f766174696f6e206e61747572616c20656e7669726f6e6d656e7420726573656172636820636f756e63696c',4320334631L,'reviewed'),
('66756e6461c3a3c2a7c3a3c2a36f20646520616d7061726f20c3a3c2a020706573717569736120646f2065737461646f2064652073c3a3c2a36f207061756c6f',4320320997L,'reviewed'),
('736369656e63652066756e6420666f722064697374696e6775697368656420796f756e67207363686f6c617273206f662068756e616e2070726f76696e6365',4320330215L,'reviewed'),
('736369656e636520616e6420746563686e6f6c6f677920706c616e2070726f6a656374206f662077656e7a686f75206d756e69636970616c697479',4320336615L,'reviewed'),
('68696768657220656475636174696f6e20636f6d6d697373696f6e2070616b697374616e',4320322799L,'reviewed'),
('752e732e206465706172746d656e74206f66206865616c746820262068756d616e207365727669636573207c206e6968207c206e6174696f6e616c2068756d616e2067656e6f6d6520726573656172636820696e73746974757465',4320337348L,'reviewed'),
('6167c3a86e636961206465206765737469c3b320642661706f733b616a75747320756e697665727369746172697320692064652072656365726361',4320334830L,'reviewed'),
('756e6974656420737461746573206465706172746d656e74206f6620646566656e7365207c20756e6974656420737461746573206e617679207c206f6666696365206f66206e6176616c207265736561726368',4320337345L,'reviewed'),
('6468207c206e6174696f6e616c20696e7374697475746520666f72206865616c7468207265736561726368',4320319990L,'reviewed'),
('686873207c206e6968207c206e6174696f6e616c2065796520696e73746974757465',4320337350L,'reviewed'),
('686f72697a6f6e2032303230',4320320300L,'programme'),
('736576656e7468206672616d65776f726b2070726f6772616d6d65',4320320300L,'programme'),
('6669667468206672616d65776f726b2070726f6772616d6d65',4320320300L,'programme'),
('6575726f7065616e20736f6369616c2066756e6420706c7573',4320320300L,'programme'),
('6575726f7065616e20756e696f6e20736576656e7468206672616d65776f726b2070726f6772616d6d65',4320320300L,'programme'),
('6575726f7065616e20636f6d6d697373696f6e206d6172696520736b6c6f646f77736b612d637572696520616374696f6e73',4320320300L,'programme'),
('7369787468206672616d65776f726b2070726f6772616d6d65',4320320300L,'programme'),
('6575206672616d65776f726b2070726f6772616d6d6520666f7220726573656172636820616e6420696e6e6f766174696f6e206d6172696520736bc5826f646f77736b612d637572696520616374696f6e73',4320320300L,'programme'),
('6575726f7065616e20756e696f6ee280997320686f72697a6f6e203230323020726573656172636820616e6420696e6e6f766174696f6e2070726f6772616d6d65',4320320300L,'programme'),
('6575726f7065616e20756e696f6e277320726573656172636820616e6420696e6e6f766174696f6e',4320320300L,'programme'),
('6575726f7065616e20756e696f6ee280997320686f72697a6f6e203230323020726573656172636820616e6420696e6e6f766174696f6e2070726f6772616d',4320320300L,'programme'),
('696e746572726567206575726f7065',4320320300L,'programme'),
('6832303230206d6172696520736b3f6f646f77736b612d637572696520616374696f6e73',4320320300L,'programme'),
('667037',4320320300L,'programme'),
('6575726f7065616e20726567696f6e616c20646576656c6f706d656e742066756e6420286572646629',4320320300L,'programme'),
('6575726f7065616e20756e696f6ee280997320686f72697a6f6e2032303230',4320320300L,'programme'),
('686f72697a6f6e2032303230206575726f7065616e20696e6e6f766174696f6e20636f756e63696c206661737420747261636b20746f20696e6e6f766174696f6e',4320320300L,'programme'),
('6575206672616d65776f726b2070726f6772616d6d6520666f7220726573656172636820616e6420696e6e6f766174696f6e206d6172696520736b6c6f646f77736b612d637572696520616374696f6e73',4320320300L,'programme'),
('6575726f7065616e20726573656172636820636f756e63696c',4320334678L,'programme'),
('6575726f7065616e20726573656172636820636f756e63696c202865726329',4320334678L,'programme'),
('6832303230206575726f7065616e20726573656172636820636f756e63696c202865726329',4320334678L,'programme')
AS a(name_hex,funder_id,rule);
END IF;
END;


In [ ]:
%sql
BEGIN
IF :phase = 'stage' THEN
-- name_twin_merges: mirrors only the reviewed merge rows in edits.csv.
-- All 59 reviewed merges are deferred; this release preserves every owner.
CREATE TABLE IDENTIFIER(:release_prefix || 'merges') AS
SELECT CAST(NULL AS BIGINT) loser_id,CAST(NULL AS BIGINT) survivor_id WHERE false;
END IF;
END;


In [ ]:
%sql
BEGIN
IF :phase = 'stage' THEN
SELECT assert_true(count(*)=0 AND count_if(l.funder_id IS NULL OR w.funder_id IS NULL OR l.merge_into_id IS NOT NULL OR w.merge_into_id IS NOT NULL OR (l.ror_id IS NOT NULL AND w.ror_id IS NOT NULL AND l.ror_id<>w.ror_id))=0,'merge preconditions changed')
FROM IDENTIFIER(:release_prefix || 'merges') m
LEFT JOIN IDENTIFIER(:release_prefix || 'catalogue_before') l ON l.funder_id=m.loser_id
LEFT JOIN IDENTIFIER(:release_prefix || 'catalogue_before') w ON w.funder_id=m.survivor_id;
SELECT assert_true(count(*)=0,'incoming redirects require renewed merge review') FROM IDENTIFIER(:release_prefix || 'catalogue_before') f JOIN IDENTIFIER(:release_prefix || 'merges') m ON f.merge_into_id=m.loser_id;
CREATE TABLE IDENTIFIER(:release_prefix || 'catalogue_after') AS
SELECT f.* EXCEPT(merge_into_id),coalesce(m.survivor_id,f.merge_into_id) merge_into_id
FROM IDENTIFIER(:release_prefix || 'catalogue_before') f
LEFT JOIN IDENTIFIER(:release_prefix || 'merges') m ON m.loser_id=f.funder_id;
CREATE TABLE IDENTIFIER(:release_prefix || 'identifiers') AS
SELECT 'ror' id_type,ror_id id_value,funder_id,funder_id origin_funder_id
FROM IDENTIFIER(:release_prefix || 'catalogue_before') WHERE ror_id IS NOT NULL;
CREATE TABLE IDENTIFIER(:release_prefix || 'registry_after') AS SELECT f.funder_id AS funder_id,CASE WHEN m.loser_id IS NOT NULL THEN 'merged' ELSE f.status END AS status,coalesce(m.survivor_id,f.merged_into_id) AS merged_into_id,f.ror_id AS ror_id,f.ror_primary AS ror_primary,f.parent_id AS parent_id,f.display_name AS display_name,f.alt_names AS alt_names,f.country_code AS country_code,f.homepage_url AS homepage_url,f.wikidata_id AS wikidata_id,f.description AS description,f.image_url AS image_url,f.image_thumbnail_url AS image_thumbnail_url,f.created_date AS created_date,CASE WHEN m.loser_id IS NOT NULL THEN current_timestamp() ELSE f.updated_date END AS updated_date,f.created_by AS created_by,CASE WHEN m.loser_id IS NOT NULL THEN :release_prefix ELSE f.run_id END AS run_id,f.mint_key AS mint_key FROM IDENTIFIER(:release_prefix || 'registry_before') f LEFT JOIN IDENTIFIER(:release_prefix || 'merges') m ON f.funder_id=m.loser_id OR f.merged_into_id=m.loser_id;
CREATE TABLE IDENTIFIER(:release_prefix || 'identifiers_after') AS SELECT f.id_type AS id_type,f.id_value AS id_value,coalesce(m.survivor_id,f.funder_id) AS funder_id,f.origin_funder_id AS origin_funder_id,f.is_primary AS is_primary,f.source AS source,f.added_at AS added_at,CASE WHEN m.loser_id IS NOT NULL THEN :release_prefix ELSE f.run_id END AS run_id FROM IDENTIFIER(:release_prefix || 'identifiers_before') f LEFT JOIN IDENTIFIER(:release_prefix || 'merges') m ON f.funder_id=m.loser_id AND f.id_type NOT IN ('name','ror_cleared');
SELECT assert_true(count(*)=0,'duplicate identifier key') FROM (SELECT id_type,id_value FROM IDENTIFIER(:release_prefix || 'identifiers_before') GROUP BY ALL HAVING count(*)<>1);
END IF;
END;


In [ ]:
%sql
BEGIN
IF :phase = 'stage' THEN
-- name_lookup: ROR/exact remain ahead of hierarchy, reviewed aliases, twins, programmes.
-- All-ROR names and broad normalization are ONLY used for pipe-delimited leaves.
-- No country/fuzzy matching and no nm1012_jev_* inputs: Jev requires a new gate.
CREATE TABLE IDENTIFIER(:release_prefix || 'lookup_rows') AS
WITH f AS (SELECT f.*, CASE WHEN m.loser_id IS NOT NULL THEN f.funder_id
 ELSE coalesce(f.merge_into_id,f.funder_id) END canonical_id
 FROM IDENTIFIER(:release_prefix || 'catalogue_after') f
 LEFT JOIN IDENTIFIER(:release_prefix || 'merges') m ON m.loser_id=f.funder_id),
alias_variants AS (
 SELECT canonical_id,lower(trim(nm)) k FROM f
 LATERAL VIEW explode(array_union(array(display_name),coalesce(from_json(alternate_titles,'array<string>'),array()))) a AS nm
), aliases AS (SELECT k,array_sort(collect_set(canonical_id)) targets FROM alias_variants WHERE k<>'' GROUP BY k),
keep_variants AS (
 SELECT f.canonical_id,lower(trim(k.name)) k FROM openalex.common.funder_names_keep k
 JOIN f ON f.funder_id=try_cast(regexp_extract(k.id,'([0-9]+)$',1) AS BIGINT)
), keeps AS (SELECT k,array_sort(collect_set(canonical_id)) targets FROM keep_variants WHERE k<>'' GROUP BY k),
names AS (
 SELECT k,targets FROM keeps
 UNION ALL SELECT a.k,a.targets FROM aliases a LEFT ANTI JOIN keeps k ON a.k=k.k
),
prod_ror_variants AS (
 SELECT canonical_id,lower(regexp_extract(trim(ror_id),r'(?i)^(?:https?://ror\.org/)?([0-9a-z]{9})/?$',1)) k FROM f
), registry_ror_variants AS (
 SELECT coalesce(f.canonical_id,r.funder_id) canonical_id,lower(regexp_extract(trim(r.id_value),r'(?i)^(?:https?://ror\.org/)?([0-9a-z]{9})/?$',1)) k
 FROM IDENTIFIER(:release_prefix || 'identifiers') r
 LEFT JOIN IDENTIFIER(:release_prefix || 'merges') tm ON tm.loser_id=r.origin_funder_id
 LEFT JOIN f ON f.funder_id=CASE WHEN tm.loser_id IS NOT NULL THEN r.origin_funder_id ELSE r.funder_id END WHERE r.id_type='ror'
), ror_variants AS (
 SELECT * FROM registry_ror_variants
 UNION ALL
 SELECT p.* FROM prod_ror_variants p LEFT ANTI JOIN registry_ror_variants r ON p.k=r.k
), rors AS (SELECT k,array_sort(collect_set(canonical_id)) targets FROM ror_variants WHERE k<>'' GROUP BY k),
-- (i) Every ROR name type is retained for candidate construction. A single
-- globally unique target is required. Standalone all-ROR matching failed 99%.
ror_names AS (
 SELECT lower(trim(n.value)) k,target funder_id
 FROM openalex.institutions.ror x JOIN rors r
 ON lower(regexp_extract(x.id,'([0-9a-z]{9})/*$',1))=r.k
 LATERAL VIEW explode(x.names) a AS n LATERAL VIEW explode(r.targets) b AS target
 WHERE trim(n.value)<>''
), dictionary AS (
 SELECT k,target funder_id FROM names LATERAL VIEW explode(targets) e AS target
 UNION ALL SELECT k,funder_id FROM ror_names
), normalized_variants AS (
 -- (ii) This normalization is used for hierarchy leaves only in the accepted
 -- ladder. Applying it to arbitrary whole strings failed the 99% criterion.
 SELECT trim(regexp_replace(regexp_replace(regexp_replace(translate(lower(replace(replace(replace(k,'&amp;','&'),'&apos;',char(39)),'&#39;',char(39))), 'áàâäãåāăąçćčďđéèêëēėęěíìîïīıłñńňóòôöõøōőřśšşťúùûüūůűýÿžźż', 'aaaaaaaaacccddeeeeeeeeiiiiiilnnnoooooooorssstuuuuuuuyyzzz'), '&', ' and '), r'[^\p{L}\p{N}]+', ' '), r'(^| )the( |$)', ' ')) nk,funder_id FROM dictionary
), normalized_names AS (
 SELECT nk,collect_set(funder_id) targets FROM normalized_variants
 WHERE length(nk)>=4 GROUP BY nk
),
twin_names AS (
 SELECT n.k,collect_set(coalesce(m.survivor_id,t)) targets,count_if(m.loser_id IS NOT NULL) changed
 FROM (SELECT k,explode(targets) t FROM names) n
 LEFT JOIN IDENTIFIER(:release_prefix || 'merges') m ON m.loser_id=t
 GROUP BY n.k
), unique_rules AS (
 SELECT 'exact' rule,k name_key,get(targets,0) funder_id FROM names WHERE size(targets)=1
 UNION ALL SELECT 'hierarchy',nk,get(targets,0) FROM normalized_names WHERE size(targets)=1
 UNION ALL SELECT rule,name_key,funder_id FROM IDENTIFIER(:release_prefix || 'aliases')
 UNION ALL SELECT 'twin',k,get(targets,0) FROM twin_names WHERE size(targets)=1 AND changed>0
)
SELECT u.rule,u.name_key,coalesce(f.merge_into_id,f.funder_id) funder_id
FROM unique_rules u
JOIN IDENTIFIER(:release_prefix || 'catalogue_after') f ON f.funder_id=u.funder_id
JOIN IDENTIFIER(:release_prefix || 'catalogue_after') target ON target.funder_id=coalesce(f.merge_into_id,f.funder_id)
WHERE target.merge_into_id IS NULL;
END IF;
END;


In [ ]:
%sql
BEGIN
IF :phase = 'stage' THEN
CREATE TABLE IDENTIFIER(:release_prefix || 'lookup') AS
SELECT l.*, :release_prefix release_id,s.catalogue_signature FROM IDENTIFIER(:release_prefix || 'lookup_rows') l
CROSS JOIN (SELECT concat(count(*), ':', cast(sum(cast(xxhash64(funder_id,display_name,alternate_titles,ror_id,doi,crossref_id,merge_into_id) AS DECIMAL(38,0))) AS STRING)) catalogue_signature FROM IDENTIFIER(:release_prefix || 'catalogue_after')) s;
SELECT assert_true((SELECT count(*) FROM IDENTIFIER(:release_prefix || 'aliases'))=204,'alias count'),
 assert_true((SELECT count(*) FROM (SELECT rule,name_key FROM IDENTIFIER(:release_prefix || 'lookup') GROUP BY ALL HAVING count(*)<>1))=0,'ambiguous lookup');
SELECT assert_true(count(*)=0,'alias target absent from approved catalogue or lookup')
FROM IDENTIFIER(:release_prefix || 'aliases') a
LEFT JOIN IDENTIFIER(:release_prefix || 'catalogue_after') f ON f.funder_id=a.funder_id
LEFT JOIN IDENTIFIER(:release_prefix || 'lookup') l ON l.rule=a.rule AND l.name_key=a.name_key AND l.funder_id=coalesce(f.merge_into_id,f.funder_id)
WHERE f.funder_id IS NULL OR l.funder_id IS NULL;
END IF;
END;


In [ ]:
%sql
BEGIN
IF :phase = 'activate' THEN
-- Readers remain on main until the last publication statement. Operator keeps
-- their schedules paused during activation and restores them for the next night.
SELECT assert_true((SELECT count(*) FROM (
 (SELECT * EXCEPT(merge_into_id),merge_into_id FROM IDENTIFIER(:catalogue) EXCEPT ALL SELECT * EXCEPT(merge_into_id),merge_into_id FROM IDENTIFIER(:release_prefix || 'catalogue_before'))
 UNION ALL (SELECT * EXCEPT(merge_into_id),merge_into_id FROM IDENTIFIER(:release_prefix || 'catalogue_before') EXCEPT ALL SELECT * EXCEPT(merge_into_id),merge_into_id FROM IDENTIFIER(:catalogue))
))=0,'catalogue changed since stage: restage and reapprove');
-- No-merge switch: registry, identifier ownership and compat rows must remain exact.
-- Publish only the signed alias/name lookup after the equality gates pass.
SELECT assert_true((SELECT count(*) FROM (
 (SELECT * FROM IDENTIFIER(:registry_prefix || 'funder') EXCEPT ALL SELECT * FROM IDENTIFIER(:release_prefix || 'registry_before'))
 UNION ALL (SELECT * FROM IDENTIFIER(:release_prefix || 'registry_before') EXCEPT ALL SELECT * FROM IDENTIFIER(:registry_prefix || 'funder'))))=0,'registry changed since stage');
SELECT assert_true((SELECT count(*) FROM (
 (SELECT * FROM IDENTIFIER(:registry_prefix || 'funder_id') EXCEPT ALL SELECT * FROM IDENTIFIER(:release_prefix || 'identifiers_before'))
 UNION ALL (SELECT * FROM IDENTIFIER(:release_prefix || 'identifiers_before') EXCEPT ALL SELECT * FROM IDENTIFIER(:registry_prefix || 'funder_id'))))=0,'identifier ownership changed since stage');
SELECT assert_true(count(*)=0 AND count_if(l.status<>'active' OR l.merged_into_id IS NOT NULL OR w.status<>'active' OR l.funder_id IS NULL OR w.funder_id IS NULL)=0,'registry merge preconditions')
FROM IDENTIFIER(:release_prefix || 'merges') m
LEFT JOIN IDENTIFIER(:registry_prefix || 'funder') l ON l.funder_id=m.loser_id
LEFT JOIN IDENTIFIER(:registry_prefix || 'funder') w ON w.funder_id=m.survivor_id;
SELECT assert_true((SELECT count(*) FROM (
 (SELECT * EXCEPT(merge_into_id),merge_into_id FROM IDENTIFIER(:catalogue) EXCEPT ALL SELECT * EXCEPT(merge_into_id),merge_into_id FROM IDENTIFIER(:release_prefix || 'catalogue_after'))
 UNION ALL (SELECT * EXCEPT(merge_into_id),merge_into_id FROM IDENTIFIER(:release_prefix || 'catalogue_after') EXCEPT ALL SELECT * EXCEPT(merge_into_id),merge_into_id FROM IDENTIFIER(:catalogue))
))=0,'catalogue publication mismatch: keep readers paused');
SELECT assert_true(count(*)=0,'funder activation mismatch: keep readers paused') FROM ((SELECT * FROM IDENTIFIER(:registry_prefix || 'funder') EXCEPT ALL SELECT * FROM IDENTIFIER(:release_prefix || 'registry_after')) UNION ALL (SELECT * FROM IDENTIFIER(:release_prefix || 'registry_after') EXCEPT ALL SELECT * FROM IDENTIFIER(:registry_prefix || 'funder')));
SELECT assert_true(count(*)=0,'funder_id activation mismatch: keep readers paused') FROM ((SELECT * FROM IDENTIFIER(:registry_prefix || 'funder_id') EXCEPT ALL SELECT * FROM IDENTIFIER(:release_prefix || 'identifiers_after')) UNION ALL (SELECT * FROM IDENTIFIER(:release_prefix || 'identifiers_after') EXCEPT ALL SELECT * FROM IDENTIFIER(:registry_prefix || 'funder_id')));
CREATE OR REPLACE TABLE IDENTIFIER(:lookup) AS SELECT * FROM IDENTIFIER(:release_prefix || 'lookup');
END IF;
END;
